# HPCQC Hands-on: multiple GPUs

This is case 3 of `cudaq-from-cpu-gpu-to-qpu.ipynb`. Start this notebook in a session with **several GPUs**.

With `option="mqpu"`, each GPU acts as a separate QPU. `sample_async` sends each circuit to a GPU (`qpu_id`), and the circuits run in parallel. We run the same batch of GHZ circuits on one GPU, then on all GPUs.

In [ ]:
import time

import cudaq

SHOTS = 1000
GHZ_QUBITS = 20     # qubits per circuit
N_CIRCUITS = 8      # circuits in the batch

N_GPUS = cudaq.num_available_gpus()
print(f"GPUs visible: {N_GPUS}")


@cudaq.kernel
def ghz_state(qubit_count: int):
    q = cudaq.qvector(qubit_count)
    h(q[0])
    for i in range(1, qubit_count):
        cx(q[0], q[i])

In [ ]:
def run_ghz_batch(n_qpus):
    for i in range(n_qpus):                                   # warm-up on every GPU
        cudaq.sample_async(ghz_state, GHZ_QUBITS, shots_count=10, qpu_id=i).get()
    t0 = time.perf_counter()
    futures = [cudaq.sample_async(ghz_state, GHZ_QUBITS, shots_count=SHOTS, qpu_id=i % n_qpus)
               for i in range(N_CIRCUITS)]
    results = [f.get() for f in futures]
    return results, time.perf_counter() - t0


cudaq.set_target("nvidia", option="mqpu")
n_qpus = cudaq.get_target().num_qpus()

times = {}
for n in sorted({1, n_qpus}):
    results, times[n] = run_ghz_batch(n)
    print(f"{N_CIRCUITS} x {GHZ_QUBITS}-qubit GHZ on {n} GPU(s): {times[n]:.2f} s")
print(f"speedup: {times[1] / times[n_qpus]:.1f}x")
print(results[0])